In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/train_easy.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/train_medium.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/sample_submission.csv
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/sample_test_easy.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/test_medium.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/sample_test_medium.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/test_easy.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/test_hard.jsonl
/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge/sample_test_hard.jsonl
/kaggle/input/final-goedel-mouryesh/__results__.html
/kaggle/input/final-goedel-mouryesh/__huggingface_repos__.json
/kaggle/input/final-goedel-mouryesh/__notebook__.ipynb
/kaggle/input/final-goedel-mouryesh/__output__.json
/kaggle/input/final-goedel-mouryesh/quantized_model.pth
/k

In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    # Verified against actual PyPI metadata + source (not guessed):
    #  - unsloth 2026.8.7 requires trl!=0.19.0,<=0.24.0,>=0.18.2 and
    #    peft!=0.11.0,>=0.18.0 (pypi.org/pypi/unsloth/json).
    #  - trl==0.24.0 (top of unsloth's allowed range) requires
    #    transformers>=4.56.1 (pypi.org/pypi/trl/0.24.0/json).
    #  - peft==0.20.0's utils/constants.py has an unconditional, unguarded
    #    `from transformers import BloomPreTrainedModel` (confirmed by reading
    #    the source directly) that breaks under transformers>=5.0, where that
    #    name was dropped from transformers' top-level lazy-import table --
    #    an unfixed upstream gap, not something a version range alone avoids.
    # So the compatible window is transformers in [4.56.1, 5.0).
    import subprocess
    subprocess.run(
        ["pip", "install", "-q", "-U",
         "unsloth", "unsloth_zoo",
         "transformers>=4.56.1,<5.0",
         "trl>=0.18.2,<=0.24.0,!=0.19.0",
         "peft>=0.18.0,!=0.11.0",
         "accelerate>=0.34.1"],
        check=True,
    )
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes "accelerate>=0.34.1" xformers==0.0.29.post3 "peft>=0.18.0,!=0.11.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer "transformers>=4.56.1,<5.0"
    !pip install --no-deps unsloth

In [ ]:
import transformers, trl, peft
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


In [ ]:
import torch
from unsloth import FastLanguageModel

# === Config ===
base_model_name = "unsloth/Phi-3.5-mini-instruct"
max_seq_length = 1024      # full window -- training truncates nothing
dtype = None
load_in_4bit = True

# Gradient checkpointing recomputes activations during backward instead of
# storing them: ~8 vs ~6 units of compute (~25% slower) to keep activation
# memory low. Gradients are IDENTICAL either way -- purely a time/memory trade.
#
# Now set to False, justified by measurement rather than assumption:
#   - benchmark peak memory was 3.15 GB of 15 GB at batch 16 (3.78 GB at 32)
#   - p90 sequence length is 223 tokens, not the 1024 window -- the data is
#     far shorter than I assumed when I first argued this was OOM-risky
# So the memory this was protecting was never close to scarce, and the 25%
# is the only remaining lever big enough to get under the 12h session cap.
#
# The benchmark cell re-checks peak memory at the LONGEST sequence in the data,
# so if this setting would OOM mid-run you find out in ~2 min, not in hour 6.
# If it does flag a problem: set this back to "unsloth", or drop PER_DEVICE_BS.
USE_GRADIENT_CHECKPOINTING = False

# === Load base model fresh -- no longer continuing from the stage-1 adapter.
# That removes the dependency on the 'Final_goedel_mouryesh' input notebook
# and its ambiguous loramodel_10epnewlysaved / loramodel_10ep_newlysaved paths
# entirely, since this run attaches a brand-new LoRA instead. ===
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Same LoRA hyperparameters as the earlier stage-2 adapter (r=16, alpha=16,
# all attn + MLP projections) so this stays comparable in capacity.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# === Sanity Check ===
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Total Parameters      : {total_params:,}")
print(f"✅ Trainable Parameters  : {trainable_params:,}")
print(f"✅ % Trainable           : {100 * trainable_params / total_params:.4f}%")
print(f"✅ Gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")

assert trainable_params > 0, "No trainable params -- LoRA setup failed."

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()


In [ ]:
import os, json, glob
from datasets import Dataset, load_from_disk

# --------------- PROMPT TEMPLATE ---------------
# NOTE: inference must reproduce this template EXACTLY (minus the {output} part),
# otherwise the LoRA adapter is queried in a format it never saw during training.
# The eval notebook imports this same constant -- keep them in sync.
board_exam_prompt = """<|system|>
You are preparing medical students for board examinations (USMLE, COMLEX). Provide accurate, evidence-based answers consistent with current medical standards.<|end|>
<|user|>
This is a medical board examination question. Apply your knowledge of clinical medicine, basic sciences, and current guidelines.

### Question:
{question}

### Options:
{options}

Choose the letter corresponding to the BEST answer based on current medical knowledge and clinical practice guidelines.<|end|>
<|assistant|>
{output}<|end|>"""

# --------------- FORMATTING FUNCTION ---------------
def format_prompt_new_style(example):
    question = example["question"].strip()
    options_str = "\n".join(
        [f"{key}. {value}" for key, value in example["options"].items()]
    )
    correct_letter = example["answer"].strip()
    correct_text = example["options"][correct_letter].strip()
    # First token of the target is the bare letter -- this is what makes
    # logit-based scoring at eval time line up with what was trained.
    assistant_output = f"{correct_letter}. {correct_text}"

    prompt = board_exam_prompt.format(
        question=question,
        options=options_str,
        output=assistant_output
    )
    return {"text": prompt}

# --------------- PREPARATION FUNCTION ---------------
def load_and_format(paths, formatting_func):
    """Merge every jsonl file in `paths`, printing a per-file count so a
    silently-empty or silently-missing tier is obvious before training starts."""
    data = []
    for path in paths:
        n_before = len(data)
        with open(path, 'r') as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"⚠️ Warning: Skipping malformed line {i+1} in {path}")
        print(f"  {os.path.basename(path)}: {len(data) - n_before:,} examples")
    print(f"  TOTAL: {len(data):,} examples merged from {len(paths)} file(s)")

    dataset = Dataset.from_list(data)
    formatted_dataset = dataset.map(formatting_func, remove_columns=dataset.column_names)
    return formatted_dataset

# --------------- EXECUTION ---------------
if __name__ == "__main__":
    COMP_DIR = "/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge"
    # Discover every train_*.jsonl the competition actually provides (easy /
    # medium / hard -- whichever tiers exist) instead of hardcoding filenames.
    train_files = sorted(glob.glob(f"{COMP_DIR}/train_*.jsonl"))
    assert train_files, f"No train_*.jsonl files found under {COMP_DIR} -- is the competition data attached?"
    print(f"Found {len(train_files)} train file(s) under {COMP_DIR}:")

    output_path = "prepared_medical_dataset"
    prepared_dataset = load_and_format(train_files, format_prompt_new_style)
    prepared_dataset.save_to_disk(output_path)
    print(f"✅ Prepared dataset saved to {output_path} with {len(prepared_dataset)} examples.")

    # Explicit shuffle across the merged tiers (train_test_split below also
    # shuffles, but this makes the "merge everything, then shuffle" step
    # visible on its own rather than hidden inside the split call).
    medical_dataset = load_from_disk(output_path).shuffle(seed=42)

    # Train on everything by default -- capping this trades away real training
    # signal, so it's a last resort, not a first-line speed knob. Set to an int
    # only if the projected time (printed by the trainer cell) still doesn't fit
    # after the zero-cost speedups.
    MAX_TRAIN_EXAMPLES = None
    if MAX_TRAIN_EXAMPLES and len(medical_dataset) > MAX_TRAIN_EXAMPLES:
        print(f"⏱️  Subsampling {len(medical_dataset):,} -> {MAX_TRAIN_EXAMPLES:,} examples.")
        medical_dataset = medical_dataset.select(range(MAX_TRAIN_EXAMPLES))

    # 90/10 train/val split.
    split_dataset = medical_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]

    # Mid-training eval exists ONLY to plot a loss curve -- it never updates
    # weights, so its size has zero effect on the trained model. Running the
    # full ~16k-example val set every 200 steps was projected at ~19h of pure
    # overhead (more than training itself, and NOT counted in the progress
    # bar's ETA). A 1k random subset gives a loss curve that's just as usable
    # for monitoring. The full val split is still saved to disk below if you
    # ever want a proper final validation pass.
    EVAL_SUBSET_SIZE = 1000
    val_dataset_eval = val_dataset.select(range(min(EVAL_SUBSET_SIZE, len(val_dataset))))

    print(f"✅ Train dataset size      : {len(train_dataset):,}")
    print(f"✅ Validation (full)       : {len(val_dataset):,}  (saved to disk)")
    print(f"✅ Validation (used in run): {len(val_dataset_eval):,}  (monitoring only)")
    assert len(train_dataset) > len(val_dataset), "train split should be the larger one"

    train_dataset.save_to_disk("train_split")
    val_dataset.save_to_disk("val_split")
    print("✅ Saved train_split and val_split directories.")


In [ ]:
import numpy as np

# DIAGNOSTIC ONLY -- this cell does not change training behaviour.
#
# An earlier version of this notebook shrank the trainer's max_length to the
# p99 token length "for speed". That was wrong: unsloth reports "Padding-free
# auto-enabled", meaning batches carry only real tokens with no padding at all.
# A 1024 cap therefore costs nothing when examples are short -- it only decides
# what gets TRUNCATED. Lowering it was throwing away the tail of the data for
# ~no speed gain. Training uses the full max_seq_length and truncates nothing.
sample_n = min(2000, len(train_dataset))
sample_lengths = np.array([
    len(tokenizer(train_dataset[i]["text"], add_special_tokens=False)["input_ids"])
    for i in range(sample_n)
])

p50, p90, p99 = np.percentile(sample_lengths, [50, 90, 99])
print(f"Token length over {sample_n} sampled training examples (max_seq_length={max_seq_length}):")
print(f"  mean={sample_lengths.mean():.0f}  p50={p50:.0f}  p90={p90:.0f}  p99={p99:.0f}  max={sample_lengths.max()}")

over_cap = (sample_lengths > max_seq_length).mean() * 100
print(f"  {over_cap:.2f}% of examples exceed max_seq_length and would be truncated by the model cap itself.")
if over_cap > 1:
    print("  ⚠️  Non-trivial truncation at the model cap -- consider RAISING max_seq_length,")
    print("      which costs little under padding-free, rather than lowering it.")

# Packing concatenates short examples to fill the window. Padding-free already
# removes the padding waste that packing would target, and packing additionally
# blends multiple questions into one context window -- which can blur the
# answer-boundary signal this task depends on. So it stays off.
USE_PACKING = False
print(f"  USE_PACKING={USE_PACKING} (padding-free already removes padding waste, at no risk)")


In [ ]:
import gc, time, torch

# Two jobs:
#   1. Throughput vs batch size. Already answered by the measured numbers:
#      2.84 / 3.04 / 3.08 ex/s at batch 4 / 8 / 16, then a REGRESSION to 2.91
#      at 32. Step time is linear in batch size, so there is no fixed
#      per-forward cost left to amortise -- the GPU is saturated and batch 16
#      is the ceiling. Kept so the result stays reproducible.
#   2. SAFETY: with gradient checkpointing OFF, activation memory scales with
#      batch x seq_len. Typical batches are short (p90 ~223 tokens), but a
#      batch can still happen to contain the longest examples in the data.
#      That worst case is measured explicitly below -- a ~2 minute check
#      instead of discovering an OOM in hour 6 of a 10 hour run.
RUN_BENCHMARK = True

try:
    _p90 = int(np.percentile(sample_lengths, 90))
    _worst = int(min(max_seq_length, sample_lengths.max()))
except Exception:
    _p90, _worst = 256, max_seq_length

_bench_seqlen = int(min(max_seq_length, max(128, _p90)))
_vocab = int(getattr(getattr(model, "config", None), "vocab_size", 32000))
results = []


def bench_batch(bs, seqlen, n_steps=3, warmup=1):
    """Time fwd+bwd at this batch size / seq len. Returns None if it OOMs."""
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    try:
        ids = torch.randint(100, min(_vocab, 30000), (bs, seqlen), device="cuda")
        model.train()
        t0 = None
        for i in range(warmup + n_steps):
            if i == warmup:
                torch.cuda.synchronize(); t0 = time.time()
            out = model(input_ids=ids, labels=ids)
            out.loss.backward()
            model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        dt = (time.time() - t0) / n_steps
        return dict(bs=bs, seqlen=seqlen, s_per_step=dt, ex_s=bs / dt,
                    peak_gb=torch.cuda.max_memory_allocated() / 1e9)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        return None
    finally:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache(); gc.collect()


_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

if RUN_BENCHMARK:
    print(f"GPU total {_total_gb:.1f} GB | gradient_checkpointing={USE_GRADIENT_CHECKPOINTING}")
    print(f"\n--- throughput @ typical seq_len={_bench_seqlen} (p90) ---")
    print(f"{'batch':>6} {'sec/step':>10} {'examples/s':>12} {'peak GB':>9}")
    print("-" * 42)
    for _bs in (4, 8, 16, 32):
        r = bench_batch(_bs, _bench_seqlen)
        if r is None:
            print(f"{_bs:>6} {'OOM':>10} {'--':>12} {'--':>9}")
            break
        results.append(r)
        print(f"{r['bs']:>6} {r['s_per_step']:>10.3f} {r['ex_s']:>12.2f} {r['peak_gb']:>9.2f}")

if results:
    usable = [r for r in results if 16 % r["bs"] == 0]
    rec = max(usable, key=lambda r: r["ex_s"]) if usable else results[0]
    print(f"\n➡️  Best usable batch (must divide effective 16): {rec['bs']} "
          f"at {rec['ex_s']:.2f} ex/s")
    print(f"   Projected training for {len(train_dataset):,} examples: "
          f"~{len(train_dataset)/rec['ex_s']/3600:.1f} h")

    # ---- worst-case memory check -------------------------------------
    # Deliberately pessimistic: a full batch of the single longest example in
    # the data. Real batches are shuffled and mostly short, so if THIS fits,
    # training will not OOM on sequence length.
    print(f"\n--- WORST-CASE memory: batch={rec['bs']} x seq_len={_worst} (longest example) ---")
    w = bench_batch(rec["bs"], _worst, n_steps=1, warmup=1)
    if w is None:
        print(f"   ✗ OOM at the worst case.")
        print( "     A batch that happens to collect the longest examples would crash training.")
        print( "     FIX (either one): set USE_GRADIENT_CHECKPOINTING='unsloth' in the model")
        print(f"     cell, or lower PER_DEVICE_BS to {max(1, rec['bs']//2)} (grad_accum compensates,")
        print( "     effective batch stays 16 so training maths is unchanged).")
    else:
        headroom = _total_gb - w["peak_gb"]
        print(f"   peak {w['peak_gb']:.2f} GB of {_total_gb:.1f} GB ({headroom:.1f} GB headroom)")
        if headroom < 2.0:
            print( "   ⚠️  Under 2 GB headroom -- tight. Allocator fragmentation over a long")
            print(f"       run could still OOM. Safer: PER_DEVICE_BS={max(1, rec['bs']//2)}.")
        else:
            print( "   ✅ Safe: even the most pessimistic batch fits with room to spare,")
            print( "      so running with gradient checkpointing off will not OOM mid-run.")
        print(f"\n   ➡️  Set PER_DEVICE_BS = {rec['bs']} in the next cell "
              f"(grad_accum {16 // rec['bs']}, effective batch stays 16).")
else:
    print("Benchmark skipped -- trainer cell falls back to its default batch size.")


In [ ]:
import os
import inspect
import torch
from trl import SFTConfig, SFTTrainer

# trl has, across versions, moved dataset_text_field/packing/max_seq_length/
# dataset_num_proc into SFTConfig, renamed SFTTrainer's tokenizer= to
# processing_class=, and occasionally renamed/dropped individual TrainingArguments
# fields too. Rather than hardcode one generation's argument names, inspect the
# *actual installed* signatures, place each movable argument where this version
# expects it, and drop -- rather than crash on -- anything it doesn't know.
#
# ---- BATCH SIZE ----------------------------------------------------------
# EFFECTIVE_BATCH is what the optimiser sees and what determines the training
# maths. Held at 16 so this run stays comparable to the original config --
# same updates, same cosine schedule, same LR.
#
# MEASURED (benchmark cell): throughput plateaus at batch 16 (2.84 / 3.04 /
# 3.08 ex/s for 4 / 8 / 16) and REGRESSES at 32 (2.91). Step time is linear in
# batch size, i.e. there is no fixed per-forward cost left to amortise -- the
# GPU is already saturated. So 16 is the ceiling; larger batches do not help.
EFFECTIVE_BATCH = 16
PER_DEVICE_BS = 16
GRAD_ACCUM = max(1, EFFECTIVE_BATCH // PER_DEVICE_BS)
assert PER_DEVICE_BS * GRAD_ACCUM == EFFECTIVE_BATCH, (
    f"PER_DEVICE_BS ({PER_DEVICE_BS}) must divide EFFECTIVE_BATCH ({EFFECTIVE_BATCH}) "
    f"so the optimisation maths stays unchanged. Got effective "
    f"{PER_DEVICE_BS * GRAD_ACCUM}. Raise EFFECTIVE_BATCH deliberately (and scale "
    f"the LR) if you actually want larger optimiser steps."
)

# ---- SPEEDUPS, all quality-neutral --------------------------------------
#  - gradient checkpointing OFF (model cell): ~25% less compute. Gradients are
#    identical; it only traded time for memory that was never scarce (measured
#    peak 3.15 GB of 15 GB).
#  - eval subset (data cell): 16k examples x 46 evals was ~19h of pure
#    overhead, more than training itself and invisible in the progress bar.
#    Eval never updates weights, so 1k every 500 steps gives the same curve.
#
# group_by_length is deliberately NOT set: unsloth reports "Padding-free
# auto-enabled", so padding is already gone and length-grouping buys nothing.
# It would actively hurt here -- grouping the longest examples into one batch
# raises worst-case memory (the thing that decides whether checkpointing-off
# OOMs) and makes batch composition length-correlated rather than shuffled.
bf16_ok = torch.cuda.is_bf16_supported()
use_packing = globals().get("USE_PACKING", False)

requested_config_kwargs = dict(
    neftune_noise_alpha=5,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=max(PER_DEVICE_BS, 16),   # eval is fwd-only
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=5,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    fp16=not bf16_ok,
    bf16=bf16_ok,
    dataloader_num_workers=2,
    seed=3407,
    output_dir="outputs",
    report_to="none",
)

movable = {
    "dataset_text_field": "text",
    "max_seq_length": max_seq_length,   # full window -- nothing truncated
    "max_length": max_seq_length,       # newer trl renamed max_seq_length -> max_length
    "packing": use_packing,
    "dataset_num_proc": min(10, os.cpu_count() or 1),
    "pad_to_multiple_of": 8,
}

config_params = inspect.signature(SFTConfig.__init__).parameters
trainer_params = inspect.signature(SFTTrainer.__init__).parameters

sft_config_kwargs = {k: v for k, v in requested_config_kwargs.items() if k in config_params}
dropped_config = [k for k in requested_config_kwargs if k not in config_params]
if dropped_config:
    print(f"⚠️  This trl version's SFTConfig doesn't accept: {dropped_config} -- dropping, using its defaults instead.")

for name, value in movable.items():
    if name in config_params:
        sft_config_kwargs.setdefault(name, value)

# Use the small eval subset if the data cell built one; fall back to the full
# val split so this cell still works if run standalone.
eval_ds = globals().get("val_dataset_eval", val_dataset)

trainer_kwargs = dict(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_ds,
    args=SFTConfig(**sft_config_kwargs),
)

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    print("⚠️  Neither processing_class= nor tokenizer= found on SFTTrainer -- "
          "relying on it to infer the tokenizer from the model.")

for name, value in movable.items():
    if name in trainer_params and name not in sft_config_kwargs:
        trainer_kwargs[name] = value

trainer = SFTTrainer(**trainer_kwargs)

print("✅ SFTTrainer constructed")
print(f"   precision={'bf16' if bf16_ok else 'fp16'}  packing={use_packing}  max_length={max_seq_length} (no truncation)")
print(f"   per_device={PER_DEVICE_BS} x grad_accum={GRAD_ACCUM} = effective {EFFECTIVE_BATCH} (unchanged)")
print(f"   gradient_checkpointing={globals().get('USE_GRADIENT_CHECKPOINTING', '?')}")
print(f"   train={len(train_dataset):,}  eval={len(eval_ds):,} every {sft_config_kwargs.get('eval_steps')} steps")

# --- Wall-clock projection ----------------------------------------------
# Use the benchmark's measured throughput for the chosen batch, scaled by the
# checkpointing setting (benchmark and training must agree, or the estimate
# silently lies -- which is what produced the earlier 13.8h vs 10h confusion).
_measured = next((r["ex_s"] for r in (globals().get("results") or []) if r.get("bs") == PER_DEVICE_BS), None)
train_ex_s = _measured or 3.08
train_h = len(train_dataset) / train_ex_s / 3600

steps = len(train_dataset) // EFFECTIVE_BATCH
n_evals = max(steps // max(sft_config_kwargs.get("eval_steps", 500), 1), 1)
eval_h = n_evals * len(eval_ds) / (train_ex_s * 4) / 3600
total_h = train_h + eval_h

print(f"\n⏱️  {'measured' if _measured else 'assumed'} {train_ex_s:.2f} ex/s -> "
      f"training ~{train_h:.1f} h + eval ~{eval_h:.1f} h ({n_evals} evals) = ~{total_h:.1f} h")
print("   NOTE: unsloth's padding-free packing means real throughput is usually")
print("   BETTER than this estimate -- the benchmark used fixed-width tensors,")
print("   while training only processes real tokens. Trust the live ETA more.")
if total_h > 11:
    print("   ⚠️  Near/over Kaggle's 12h cap. Checkpoints every 500 steps mean a timeout")
    print("      can be resumed rather than lost -- or set MAX_TRAIN_EXAMPLES in the data cell.")
else:
    print("   ✅ Fits inside a 12h session with margin.")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Resume from the last checkpoint if one exists in this session's outputs/
# (e.g. after a browser disconnect or a manual stop) instead of restarting
# from step 0. This only helps within a live/reconnected Kaggle draft session --
# it does not survive a full session being torn down and recreated from
# scratch, since /kaggle/working isn't preserved across that.
import glob
_checkpoints = sorted(
    glob.glob("outputs/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[1]),
)
RESUME_FROM = _checkpoints[-1] if _checkpoints else None
print(f"Resuming from: {RESUME_FROM}" if RESUME_FROM else "No checkpoint found -- starting fresh.")

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=RESUME_FROM)

In [ ]:
import json, os

# Distinct lineage from stage-2/stage-3 (those continued from a stage-1
# adapter on medium-only data); this is a fresh LoRA trained on all merged
# tiers, so it gets its own name rather than another "base_N" increment.
ADAPTER_OUT = "loramodel_final"
model.save_pretrained(ADAPTER_OUT)   # Local saving
tokenizer.save_pretrained(ADAPTER_OUT)

# Persist the loss curve so it survives the session and can go straight into the writeup.
os.makedirs("results", exist_ok=True)
history = trainer.state.log_history
with open("results/train_log_history.json", "w") as f:
    json.dump(history, f, indent=2)

train_losses = [h for h in history if "loss" in h]
eval_losses  = [h for h in history if "eval_loss" in h]
print(f"✅ Adapter saved to {ADAPTER_OUT}")
print(f"✅ {len(train_losses)} train log points, {len(eval_losses)} eval points -> results/train_log_history.json")
if train_losses:
    print(f"   first train loss: {train_losses[0]['loss']:.4f}  ->  last: {train_losses[-1]['loss']:.4f}")
if eval_losses:
    print(f"   first eval  loss: {eval_losses[0]['eval_loss']:.4f}  ->  last: {eval_losses[-1]['eval_loss']:.4f}")


In [ ]:
import json, os
import matplotlib.pyplot as plt

# Run AFTER training (or after stopping it early -- log_history holds whatever
# was recorded up to that point, so this works on a partial run too).
# Reads the live trainer if it's in memory, else the JSON saved above.
if "trainer" in globals() and getattr(trainer, "state", None) is not None:
    history = trainer.state.log_history
else:
    with open("results/train_log_history.json") as f:
        history = json.load(f)

tr = [(h["step"], h["loss"]) for h in history if "loss" in h and "step" in h]
ev = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h and "step" in h]

fig, ax = plt.subplots(figsize=(9, 5))
if tr:
    ax.plot(*zip(*tr), lw=1, alpha=.45, label=f"train (every 10 steps, n={len(tr)})")
    # rolling mean -- per-step training loss is noisy enough to hide the trend
    w = max(1, len(tr) // 50)
    if w > 1:
        xs, ys = zip(*tr)
        smooth = [sum(ys[max(0, i - w):i + 1]) / len(ys[max(0, i - w):i + 1]) for i in range(len(ys))]
        ax.plot(xs, smooth, lw=2, label=f"train (smoothed, window={w})")
if ev:
    ax.plot(*zip(*ev), "o-", lw=2, ms=5, label=f"validation (every 500 steps, n={len(ev)})")

ax.set_xlabel("optimizer step"); ax.set_ylabel("loss")
ax.set_title("Training / validation loss")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig("results/loss_curve.png", dpi=150)
plt.show()

# The question worth answering from this curve: did validation loss actually
# keep improving, or did it flatten early? If it flattened, the later hours of
# training bought nothing -- which is a legitimate finding to report, and tells
# you a smaller MAX_TRAIN_EXAMPLES would have been just as good.
if len(ev) >= 4:
    xs, ys = zip(*ev)
    best_i = min(range(len(ys)), key=lambda i: ys[i])
    first_half_gain = ys[0] - ys[len(ys) // 2]
    second_half_gain = ys[len(ys) // 2] - ys[-1]
    print(f"validation loss: {ys[0]:.4f} (step {xs[0]}) -> {ys[-1]:.4f} (step {xs[-1]})")
    print(f"  best: {ys[best_i]:.4f} at step {xs[best_i]}"
          f"{'  <-- BEST WAS NOT THE LAST STEP (later training did not help)' if best_i < len(ys)-1 else ''}")
    print(f"  improvement in first half : {first_half_gain:+.4f}")
    print(f"  improvement in second half: {second_half_gain:+.4f}")
    if second_half_gain < first_half_gain * 0.25:
        print("  -> Diminishing returns: the back half of the run contributed little.")
        print("     Worth reporting, and worth training on less data next time.")
    else:
        print("  -> Still improving through the end -- the full dataset was justified.")
elif ev:
    print(f"only {len(ev)} eval point(s) so far -- need ~4+ to judge the trend.")
else:
    print("No eval points recorded yet.")


In [ ]:
import os

# SECURITY: the token used to be hardcoded here in plaintext. That old token
# (hf_Prug...) is compromised -- revoke it at https://huggingface.co/settings/tokens
# and create a new one. On Kaggle: Add-ons -> Secrets -> add HF_TOKEN, then attach it.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

# This push is optional -- the RAG notebook only needs the local ADAPTER_OUT
# folder saved in the previous cell. Skip cleanly rather than failing the run
# (and blocking "Save & Run All") when no token is configured.
if not HF_TOKEN:
    print(f"ℹ️  HF_TOKEN not found -- skipping Hub push. '{ADAPTER_OUT}' is already "
          f"saved locally, which is all the RAG notebook needs.")
else:
    HF_REPO = "mouryesh/onbase_final"
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"✅ Pushed adapter + tokenizer to {HF_REPO}")
